# Data Prep for Unsupervised Learning

In [77]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Row Granularity

In [2]:
# Creating a sample dataframe

songs_dict = {
                'Customer': ['Aria', 'Aria', 'Aria', 'Chord', 'Chord', 'Harmony', 'Harmony', 'Harmony', 'Melody', 'Reed'],
                'Genre': ['Pop', 'Indie', 'Rock', 'Pop', 'Indie', 'Pop', 'Indie', 'Rock', 'Rock', 'Rock'],
                '# Songs': [50, 48, 1, 15, 36, 10, 5, 3, 2, 5]
             }

df = pd.DataFrame(songs_dict)
df

,Customer,Genre,# Songs
0,Aria,Pop,50
1,Aria,Indie,48
2,Aria,Rock,1
3,Chord,Pop,15
4,Chord,Indie,36
5,Harmony,Pop,10
6,Harmony,Indie,5
7,Harmony,Rock,3
8,Melody,Rock,2
9,Reed,Rock,5


### a. Group By

In [3]:
df.groupby("Customer")["# Songs"].sum().reset_index()

,Customer,# Songs
0,Aria,99
1,Chord,51
2,Harmony,18
3,Melody,2
4,Reed,5


### b. Pivot

In [4]:
df

,Customer,Genre,# Songs
0,Aria,Pop,50
1,Aria,Indie,48
2,Aria,Rock,1
3,Chord,Pop,15
4,Chord,Indie,36
5,Harmony,Pop,10
6,Harmony,Indie,5
7,Harmony,Rock,3
8,Melody,Rock,2
9,Reed,Rock,5


In [5]:
customers_genres = df.pivot(index="Customer",
                             columns="Genre",
                             values="# Songs").fillna(0).reset_index()

In [6]:
customers_genres

Genre,Customer,Indie,Pop,Rock
0,Aria,48.0,50.0,1.0
1,Chord,36.0,15.0,0.0
2,Harmony,5.0,10.0,3.0
3,Melody,0.0,0.0,2.0
4,Reed,0.0,0.0,5.0


## 2. Non-Null and Numeric Columns

In [7]:
customers_raw = pd.read_csv("../Data/customers.csv")
customers_raw

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Harmony,26.0,NaN,"$120,000",4/25/23,No,Graduate School
3,Melody,47.0,NaN,"$450,000",5/5/23,No,College
4,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
5,Selena,NaN,1.0,"$62,000",8/26/23,No,College
6,Stefani,NaN,NaN,"$81,000",9/24/23,No,College
7,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


In [8]:
customers = customers_raw.copy()
customers

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Harmony,26.0,NaN,"$120,000",4/25/23,No,Graduate School
3,Melody,47.0,NaN,"$450,000",5/5/23,No,College
4,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
5,Selena,NaN,1.0,"$62,000",8/26/23,No,College
6,Stefani,NaN,NaN,"$81,000",9/24/23,No,College
7,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


In [9]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             8 non-null      object 
 1   Age              6 non-null      float64
 2   Followers        5 non-null      float64
 3   Income           8 non-null      object 
 4   Sign Up Date     8 non-null      object 
 5   Discount         8 non-null      object 
 6   Education Level  8 non-null      object 
dtypes: float64(2), object(5)
memory usage: 580.0+ bytes


In [10]:
customers.isna().sum()

Name               0
Age                2
Followers          3
Income             0
Sign Up Date       0
Discount           0
Education Level    0
dtype: int64

In [11]:
customers[customers.isna().any(axis=1)]

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
2,Harmony,26.0,NaN,"$120,000",4/25/23,No,Graduate School
3,Melody,47.0,NaN,"$450,000",5/5/23,No,College
5,Selena,NaN,1.0,"$62,000",8/26/23,No,College
6,Stefani,NaN,NaN,"$81,000",9/24/23,No,College


### a. Handling Null Values

#### i. Dropping Missing Values

In [12]:
customers.dropna().reset_index(drop=True)

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
3,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


In [13]:
customers_dropped = customers.dropna().reset_index(drop=True)
customers_dropped

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
3,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


#### ii. Imputing Missing Values

In [14]:
customers

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Harmony,26.0,NaN,"$120,000",4/25/23,No,Graduate School
3,Melody,47.0,NaN,"$450,000",5/5/23,No,College
4,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
5,Selena,NaN,1.0,"$62,000",8/26/23,No,College
6,Stefani,NaN,NaN,"$81,000",9/24/23,No,College
7,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


In [15]:
round(customers["Age"].fillna(customers["Age"].median()))

0    25.0
1    19.0
2    26.0
3    47.0
4    52.0
5    30.0
6    30.0
7    33.0
Name: Age, dtype: float64

In [16]:
customers["Followers"].fillna(0)

0     0.0
1    12.0
2     0.0
3     0.0
4     0.0
5     1.0
6     0.0
7    52.0
Name: Followers, dtype: float64

In [17]:
customers["Age"] = round(customers["Age"].fillna(customers["Age"].median()))
customers["Followers"] = customers["Followers"].fillna(0)
customers

,Name,Age,Followers,Income,Sign Up Date,Discount,Education Level
0,Aria,25.0,0.0,"$45,000",5/18/23,Yes,College
1,Chord,19.0,12.0,"$28,000",8/23/23,Yes,High School
2,Harmony,26.0,0.0,"$120,000",4/25/23,No,Graduate School
3,Melody,47.0,0.0,"$450,000",5/5/23,No,College
4,Reed,52.0,0.0,"$75,000",6/14/23,Yes,High School
5,Selena,30.0,1.0,"$62,000",8/26/23,No,College
6,Stefani,30.0,0.0,"$81,000",9/24/23,No,College
7,Taylor,33.0,52.0,"$60,000",9/8/23,No,High School


### b. Converting Data Types

In [18]:
customers.dtypes

Name                object
Age                float64
Followers          float64
Income              object
Sign Up Date        object
Discount            object
Education Level     object
dtype: object

#### i. Converting to Numeric

In [19]:
customers["Income"].str.replace("$", "").str.replace(",","")

0     45000 
1     28000 
2    120000 
3    450000 
4     75000 
5     62000 
6     81000 
7     60000 
Name: Income, dtype: object

In [20]:
pd.to_numeric(customers["Income"].str.replace("$", "").str.replace(",",""))

0     45000
1     28000
2    120000
3    450000
4     75000
5     62000
6     81000
7     60000
Name: Income, dtype: int64

In [21]:
customers["Income"] = pd.to_numeric(customers["Income"].str.replace("$", "").str.replace(",",""))
customers["Income"]

0     45000
1     28000
2    120000
3    450000
4     75000
5     62000
6     81000
7     60000
Name: Income, dtype: int64

In [22]:
customers.dtypes

Name                object
Age                float64
Followers          float64
Income               int64
Sign Up Date        object
Discount            object
Education Level     object
dtype: object

In [23]:
customers["Age"] = customers["Age"].astype(int)
customers["Followers"] = customers["Followers"].astype(int)

In [24]:
customers.dtypes

Name               object
Age                 int64
Followers           int64
Income              int64
Sign Up Date       object
Discount           object
Education Level    object
dtype: object

#### ii. Converting to DateTime

In [25]:
pd.to_datetime(customers["Sign Up Date"], format="%m/%d/%y")

0   2023-05-18
1   2023-08-23
2   2023-04-25
3   2023-05-05
4   2023-06-14
5   2023-08-26
6   2023-09-24
7   2023-09-08
Name: Sign Up Date, dtype: datetime64[ns]

In [26]:
customers["Sign Up Date"] = pd.to_datetime(customers["Sign Up Date"], format="%m/%d/%y")

In [27]:
customers.dtypes

Name                       object
Age                         int64
Followers                   int64
Income                      int64
Sign Up Date       datetime64[ns]
Discount                   object
Education Level            object
dtype: object

#### iii. Extracting DateTime Components

In [28]:
customers["Sign Up Month"] = customers["Sign Up Date"].dt.month

In [29]:
customers["Sign Up Day"] = customers["Sign Up Date"].dt.dayofweek

### c. Conditional Logic

In [34]:
customers["Discount"] = np.where(customers["Discount"] == "Yes", 1, 0)
customers["Discount"]

0    1
1    1
2    0
3    0
4    1
5    0
6    0
7    0
Name: Discount, dtype: int64

In [35]:
customers

,Name,Age,Followers,Income,Discount,Education Level,Sign Up Month,Sign Up Day
0,Aria,25,0,45000,1,College,5,3
1,Chord,19,12,28000,1,High School,8,2
2,Harmony,26,0,120000,0,Graduate School,4,1
3,Melody,47,0,450000,0,College,5,4
4,Reed,52,0,75000,1,High School,6,2
5,Selena,30,1,62000,0,College,8,5
6,Stefani,30,0,81000,0,College,9,6
7,Taylor,33,52,60000,0,High School,9,4


### d. Dummy Variables

In [38]:
dummy_edu = pd.get_dummies(customers["Education Level"]).astype(int)

In [39]:
dummy_edu

,College,Graduate School,High School
0,1,0,0
1,0,0,1
2,0,1,0
3,1,0,0
4,0,0,1
5,1,0,0
6,1,0,0
7,0,0,1


In [41]:
customers = pd.concat([customers, dummy_edu], axis=1)

In [42]:
customers

,Name,Age,Followers,Income,Discount,Education Level,Sign Up Month,Sign Up Day,College,Graduate School,High School
0,Aria,25,0,45000,1,College,5,3,1,0,0
1,Chord,19,12,28000,1,High School,8,2,0,0,1
2,Harmony,26,0,120000,0,Graduate School,4,1,0,1,0
3,Melody,47,0,450000,0,College,5,4,1,0,0
4,Reed,52,0,75000,1,High School,6,2,0,0,1
5,Selena,30,1,62000,0,College,8,5,1,0,0
6,Stefani,30,0,81000,0,College,9,6,1,0,0
7,Taylor,33,52,60000,0,High School,9,4,0,0,1


In [47]:
customers = customers.drop(columns=["Education Level"])

In [48]:
customers

,Name,Age,Followers,Income,Discount,Sign Up Month,Sign Up Day,College,Graduate School,High School
0,Aria,25,0,45000,1,5,3,1,0,0
1,Chord,19,12,28000,1,8,2,0,0,1
2,Harmony,26,0,120000,0,4,1,0,1,0
3,Melody,47,0,450000,0,5,4,1,0,0
4,Reed,52,0,75000,1,6,2,0,0,1
5,Selena,30,1,62000,0,8,5,1,0,0
6,Stefani,30,0,81000,0,9,6,1,0,0
7,Taylor,33,52,60000,0,9,4,0,0,1


## 3. Feature Engineering

### a. Applying Calculations

In [49]:
# the data from setting the row granularity, with a few more customers
songs_genres_dict = {'Customer': ['Aria', 'Chord', 'Harmony', 'Melody', 'Reed', 'Selena', 'Stefani', 'Taylor'],
                     '# Songs': [99, 51, 18, 2, 5, 60, 15, 121],
                     'Indie': [48, 36, 5, 0, 0, 20, 2, 19],
                     'Pop': [50, 15, 10, 0, 0, 20, 5, 89],
                     'Rock': [1, 0, 3, 2, 5, 20, 8, 13]}

songs_genres = pd.DataFrame(songs_genres_dict)
songs_genres

,Customer,# Songs,Indie,Pop,Rock
0,Aria,99,48,50,1
1,Chord,51,36,15,0
2,Harmony,18,5,10,3
3,Melody,2,0,0,2
4,Reed,5,0,0,5
5,Selena,60,20,20,20
6,Stefani,15,2,5,8
7,Taylor,121,19,89,13


In [50]:
customers

,Name,Age,Followers,Income,Discount,Sign Up Month,Sign Up Day,College,Graduate School,High School
0,Aria,25,0,45000,1,5,3,1,0,0
1,Chord,19,12,28000,1,8,2,0,0,1
2,Harmony,26,0,120000,0,4,1,0,1,0
3,Melody,47,0,450000,0,5,4,1,0,0
4,Reed,52,0,75000,1,6,2,0,0,1
5,Selena,30,1,62000,0,8,5,1,0,0
6,Stefani,30,0,81000,0,9,6,1,0,0
7,Taylor,33,52,60000,0,9,4,0,0,1


In [53]:
model_df = pd.concat([customers, songs_genres], axis=1).drop(columns=["Customer"])

In [54]:
model_df

,Name,Age,Followers,Income,Discount,Sign Up Month,Sign Up Day,College,Graduate School,High School,# Songs,Indie,Pop,Rock
0,Aria,25,0,45000,1,5,3,1,0,0,99,48,50,1
1,Chord,19,12,28000,1,8,2,0,0,1,51,36,15,0
2,Harmony,26,0,120000,0,4,1,0,1,0,18,5,10,3
3,Melody,47,0,450000,0,5,4,1,0,0,2,0,0,2
4,Reed,52,0,75000,1,6,2,0,0,1,5,0,0,5
5,Selena,30,1,62000,0,8,5,1,0,0,60,20,20,20
6,Stefani,30,0,81000,0,9,6,1,0,0,15,2,5,8
7,Taylor,33,52,60000,0,9,4,0,0,1,121,19,89,13


In [55]:
model_df["Pop"]

0    50
1    15
2    10
3     0
4     0
5    20
6     5
7    89
Name: Pop, dtype: int64

In [56]:
model_df["# Songs"]

0     99
1     51
2     18
3      2
4      5
5     60
6     15
7    121
Name: # Songs, dtype: int64

In [59]:
model_df["pop_pct"] = round(model_df["Pop"] / model_df["# Songs"], 2)
model_df

,Name,Age,Followers,Income,Discount,Sign Up Month,Sign Up Day,College,Graduate School,High School,# Songs,Indie,Pop,Rock,pop_pct
0,Aria,25,0,45000,1,5,3,1,0,0,99,48,50,1,0.51
1,Chord,19,12,28000,1,8,2,0,0,1,51,36,15,0,0.29
2,Harmony,26,0,120000,0,4,1,0,1,0,18,5,10,3,0.56
3,Melody,47,0,450000,0,5,4,1,0,0,2,0,0,2,0.00
4,Reed,52,0,75000,1,6,2,0,0,1,5,0,0,5,0.00
5,Selena,30,1,62000,0,8,5,1,0,0,60,20,20,20,0.33
6,Stefani,30,0,81000,0,9,6,1,0,0,15,2,5,8,0.33
7,Taylor,33,52,60000,0,9,4,0,0,1,121,19,89,13,0.74


### b. Binning Values

In [62]:
model_df["Weekend"] = np.where(customers["Sign Up Day"].isin([5, 6]), 1, 0)

In [63]:
model_df

,Name,Age,Followers,Income,Discount,Sign Up Month,Sign Up Day,College,Graduate School,High School,# Songs,Indie,Pop,Rock,pop_pct,Weekend
0,Aria,25,0,45000,1,5,3,1,0,0,99,48,50,1,0.51,0
1,Chord,19,12,28000,1,8,2,0,0,1,51,36,15,0,0.29,0
2,Harmony,26,0,120000,0,4,1,0,1,0,18,5,10,3,0.56,0
3,Melody,47,0,450000,0,5,4,1,0,0,2,0,0,2,0.00,0
4,Reed,52,0,75000,1,6,2,0,0,1,5,0,0,5,0.00,0
5,Selena,30,1,62000,0,8,5,1,0,0,60,20,20,20,0.33,1
6,Stefani,30,0,81000,0,9,6,1,0,0,15,2,5,8,0.33,1
7,Taylor,33,52,60000,0,9,4,0,0,1,121,19,89,13,0.74,0


In [64]:
model_df = model_df.drop(columns=["Sign Up Day"])

In [65]:
model_df

,Name,Age,Followers,Income,Discount,Sign Up Month,College,Graduate School,High School,# Songs,Indie,Pop,Rock,pop_pct,Weekend
0,Aria,25,0,45000,1,5,1,0,0,99,48,50,1,0.51,0
1,Chord,19,12,28000,1,8,0,0,1,51,36,15,0,0.29,0
2,Harmony,26,0,120000,0,4,0,1,0,18,5,10,3,0.56,0
3,Melody,47,0,450000,0,5,1,0,0,2,0,0,2,0.00,0
4,Reed,52,0,75000,1,6,0,0,1,5,0,0,5,0.00,0
5,Selena,30,1,62000,0,8,1,0,0,60,20,20,20,0.33,1
6,Stefani,30,0,81000,0,9,1,0,0,15,2,5,8,0.33,1
7,Taylor,33,52,60000,0,9,0,0,1,121,19,89,13,0.74,0


### c. Proxy Variables

In [66]:
# external data with the average temperature in F each month in Chicago
avg_temp_dict = {'Month': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
                 'Avg_Temp': [32, 36, 45, 56, 66, 77, 82, 81, 74, 62, 50, 37]}

avg_temp = pd.DataFrame(avg_temp_dict)
avg_temp

,Month,Avg_Temp
0,1,32
1,2,36
2,3,45
3,4,56
4,5,66
5,6,77
6,7,82
7,8,81
8,9,74
9,10,62


In [68]:
model_df = pd.merge(model_df, avg_temp, left_on="Sign Up Month", right_on="Month")

In [69]:
model_df = model_df.drop(columns="Sign Up Month")

In [70]:
model_df

,Name,Age,Followers,Income,Discount,College,Graduate School,High School,# Songs,Indie,Pop,Rock,pop_pct,Weekend,Month,Avg_Temp
0,Aria,25,0,45000,1,1,0,0,99,48,50,1,0.51,0,5,66
1,Chord,19,12,28000,1,0,0,1,51,36,15,0,0.29,0,8,81
2,Harmony,26,0,120000,0,0,1,0,18,5,10,3,0.56,0,4,56
3,Melody,47,0,450000,0,1,0,0,2,0,0,2,0.00,0,5,66
4,Reed,52,0,75000,1,0,0,1,5,0,0,5,0.00,0,6,77
5,Selena,30,1,62000,0,1,0,0,60,20,20,20,0.33,1,8,81
6,Stefani,30,0,81000,0,1,0,0,15,2,5,8,0.33,1,9,74
7,Taylor,33,52,60000,0,0,0,1,121,19,89,13,0.74,0,9,74


## 4. Feature Selection

### a. Exclude ID Columns

In [71]:
names = model_df.Name

In [72]:
names

0       Aria
1      Chord
2    Harmony
3     Melody
4       Reed
5     Selena
6    Stefani
7     Taylor
Name: Name, dtype: object

In [74]:
model_df = model_df.drop(columns=["Name"])

In [75]:
model_df

,Age,Followers,Income,Discount,College,Graduate School,High School,# Songs,Indie,Pop,Rock,pop_pct,Weekend,Month,Avg_Temp
0,25,0,45000,1,1,0,0,99,48,50,1,0.51,0,5,66
1,19,12,28000,1,0,0,1,51,36,15,0,0.29,0,8,81
2,26,0,120000,0,0,1,0,18,5,10,3,0.56,0,4,56
3,47,0,450000,0,1,0,0,2,0,0,2,0.00,0,5,66
4,52,0,75000,1,0,0,1,5,0,0,5,0.00,0,6,77
5,30,1,62000,0,1,0,0,60,20,20,20,0.33,1,8,81
6,30,0,81000,0,1,0,0,15,2,5,8,0.33,1,9,74
7,33,52,60000,0,0,0,1,121,19,89,13,0.74,0,9,74


In [76]:
model_subset = model_df[["Age", "# Songs", "pop_pct"]]
model_subset

,Age,# Songs,pop_pct
0,25,99,0.51
1,19,51,0.29
2,26,18,0.56
3,47,2,0.00
4,52,5,0.00
5,30,60,0.33
6,30,15,0.33
7,33,121,0.74


## 5. Feature Scaling

In [80]:
from sklearn.preprocessing import MinMaxScaler

### a. Normalization

In [81]:
mm_scaler = MinMaxScaler()

In [85]:
mm_scaler.fit_transform(model_subset)
pd.DataFrame(normalized, columns=model_subset.columns)

,Age,# Songs,pop_pct
0,0.181818,0.815126,0.689189
1,0.000000,0.411765,0.391892
2,0.212121,0.134454,0.756757
3,0.848485,0.000000,0.000000
4,1.000000,0.025210,0.000000
5,0.333333,0.487395,0.445946
6,0.333333,0.109244,0.445946
7,0.424242,1.000000,1.000000


### b. Standardization

In [87]:
from sklearn.preprocessing import StandardScaler

In [91]:
std_scaler = StandardScaler()
standardized = std_scaler.fit_transform(model_subset)
pd.DataFrame(standardized, columns=model_subset.columns)

,Age,# Songs,pop_pct
0,-0.737468,1.257265,0.680015
1,-1.308412,0.110496,-0.226672
2,-0.642311,-0.677908,0.886080
3,1.355990,-1.060164,-1.421850
4,1.831776,-0.988491,-1.421850
5,-0.261682,0.325515,-0.061820
6,-0.261682,-0.749581,-0.061820
7,0.023789,1.782868,1.627915
